<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/01_environment_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Environment, Toolchain, and Baseline RAG Pipeline

**Goal:** Install and configure all four evaluation tools (RAGAS, DeepEval, Langfuse, Promptfoo), establish Langfuse tracing from day one, and build the baseline RAG pipeline that all subsequent phases evaluate.

**Tools:** RAGAS v0.2+, DeepEval, Langfuse v4, Promptfoo (npm), Chroma, Gemini API, Claude API

**Drive path:** /content/drive/MyDrive/python-ai-governance-p2/data/

**Narrative:** Project 1 built the governance system. Project 2 evaluates whether it holds up under production conditions. Gemini (gemini-flash-latest) is the system under test throughout. Claude (claude-sonnet-4-6) is the cross-model evaluator and LLM judge. This implements the same-family observer bias finding from Project 1 Phase 7 structurally: a model cannot reliably audit a system it shares blind spots with.

**SIMULATED_OUTPUT flag:** Set to True throughout this notebook. All code is complete and runnable. When API credits are available, set SIMULATED_OUTPUT = False in Cell 4 and every cell runs live with identical logic. Simulated outputs match the exact schema of real API responses.

**Date:** July 2026

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"
os.makedirs(DRIVE_PATH, exist_ok=True)

print(f"Drive mounted.")
print(f"Project 2 data path: {DRIVE_PATH}")

Mounted at /content/drive
Drive mounted.
Project 2 data path: /content/drive/MyDrive/python-ai-governance-p2/data/


In [2]:
# Run this cell once per Colab session.
# Restart the runtime after installation before continuing.

!pip install ragas==0.2.0 deepeval langfuse chromadb \
    google-generativeai anthropic sentence-transformers --quiet

print("All packages installed. Restart runtime now, then continue from Cell 4.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/

In [3]:
# Colab's default Node.js is too old for current Promptfoo.
# This cell upgrades Node to v22 (LTS) before installing Promptfoo.

# Step 1: Install nvm (Node Version Manager)
!curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash

# Step 2: Load nvm and install Node 22
import subprocess
nvm_cmd = """
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm install 22
nvm use 22
node --version
"""
result = subprocess.run(["bash", "-c", nvm_cmd], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-500:] if result.stderr else "")

# Step 3: Install Promptfoo using the upgraded Node
promptfoo_cmd = """
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm use 22
npm install -g promptfoo@latest
promptfoo --version
"""
result2 = subprocess.run(["bash", "-c", promptfoo_cmd], capture_output=True, text=True)
print(result2.stdout)
print(result2.stderr[-300:] if result2.stderr else "")

# Step 4: Set the environment variable
import os
os.environ["PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION"] = "true"
print("Remote generation disabled.")

=> Downloading nvm from git to '/root/.nvm'
=> Cloning into '/root/.nvm'...
remote: Enumerating objects: 461, done.
remote: Counting objects: 100% (461/461), done.
remote: Compressing objects: 100% (383/383), done.
remote: Total 461 (delta 68), reused 222 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (461/461), 458.03 KiB | 13.88 MiB/s, done.
Resolving deltas: 100% (68/68), done.
* (HEAD detached at FETCH_HEAD)
  master
=> Compressing and cleaning up git repository

=> Appending nvm source string to /root/.bashrc
=> Appending bash_completion source string to /root/.bashrc
=> You currently have modules installed globally with `npm`. These will no
=> longer be linked to the active version of Node when you install a new node
=> with `nvm`; and they may (depending on how you construct your `$PATH`)
=> override the binaries of modules installed with `nvm`:

/tools/node/lib
├── @google/gemini-cli@0.50.0
├── corepack@0.31.0
└── promptfoo@0.121.19
=> If you wish to uninstall them 

In [4]:
import subprocess

def run_promptfoo(args: str, cwd: str = None) -> str:
    """Run a promptfoo command using the nvm Node v22 environment.

    All Promptfoo calls in this project go through this function
    to ensure the correct Node version is used regardless of what
    Colab's default shell resolves to.
    """
    cmd = f"""
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm use 22 --silent
promptfoo {args}
"""
    result = subprocess.run(
        ["bash", "-c", cmd],
        capture_output=True,
        text=True,
        cwd=cwd
    )
    if result.returncode != 0 and result.stderr:
        print(f"[STDERR]: {result.stderr[-500:]}")
    return result.stdout


# Verify the helper works correctly
version_output = run_promptfoo("--version")
print(f"Promptfoo via helper: {version_output.strip()}")
print("run_promptfoo() ready. All Phase 5 Promptfoo calls use this function.")

Promptfoo via helper: 0.121.19
run_promptfoo() ready. All Phase 5 Promptfoo calls use this function.


In [6]:
# SIMULATED_OUTPUT = True: uses representative mock data throughout.
# SIMULATED_OUTPUT = False: requires the following Colab secrets:
#   GOOGLE_API_KEY       (Gemini, system under test)
#   ANTHROPIC_API_KEY    (Claude, cross-model judge)
#   LANGFUSE_PUBLIC_KEY  (observability dashboard)
#   LANGFUSE_SECRET_KEY  (observability dashboard)
#
# Flip this single flag when API credits are available.
# No other changes required in any cell.

SIMULATED_OUTPUT = True

from google.colab import userdata

if not SIMULATED_OUTPUT:
    from google import genai as google_genai
    gemini_client = google_genai.Client(
        api_key=userdata.get('GOOGLE_API_KEY')
    )
    print("Gemini client initialised (system under test).")

    import anthropic
    claude_client = anthropic.Anthropic(
        api_key=userdata.get('ANTHROPIC_API_KEY')
    )
    print("Claude client initialised (cross-model judge).")

    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Langfuse client initialised.")

else:
    print("[SIMULATED] API clients not initialised.")
    print("Required Colab secrets when running live:")
    print("  GOOGLE_API_KEY       -> Gemini (system under test)")
    print("  ANTHROPIC_API_KEY    -> Claude (cross-model judge)")
    print("  LANGFUSE_PUBLIC_KEY  -> Langfuse observability")
    print("  LANGFUSE_SECRET_KEY  -> Langfuse observability")
    print()
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")

[SIMULATED] API clients not initialised.
Required Colab secrets when running live:
  GOOGLE_API_KEY       -> Gemini (system under test)
  ANTHROPIC_API_KEY    -> Claude (cross-model judge)
  LANGFUSE_PUBLIC_KEY  -> Langfuse observability
  LANGFUSE_SECRET_KEY  -> Langfuse observability

SIMULATED_OUTPUT = True


## Architectural Note: Why Gemini and Claude

Project 1 Phase 7 found that the Observer Agent returned perfect 5/5 scores on every evaluation query because it shared the same model family as the RAG Agent it was auditing. The auditor had structurally identical blind spots to the system under evaluation.

Project 2 addresses this structurally, not with a better prompt:

- **Gemini (gemini-flash-latest):** system under test. Generates responses, completes tasks, is evaluated.
- **Claude (claude-sonnet-4-6):** cross-model evaluator and LLM judge throughout every phase. Never evaluates its own outputs.

This is the same principle that makes financial audit independence non-negotiable. A different model family removes the shared blind spot. It does not remove the institutional blind spot (same operator), which is why Phase 6 documents the integrity-of-record vs fidelity-of-judgment distinction explicitly.

In [11]:
REGULATORY_DOCS = {
    "doc_001": {
        "title": "EU AI Act Article 10: Data Governance",
        "content": (
            "Article 10 requires that high-risk AI systems use training, validation "
            "and testing data subject to data governance practices. Data sets must be "
            "relevant, representative, and free of errors. Providers must examine data "
            "for possible biases. Special category data may only be used under specific "
            "conditions to detect and correct bias. Disparate impact ratios below 0.80 "
            "indicate a potential Article 10 violation."
        )
    },
    "doc_002": {
        "title": "EU AI Act Article 14: Human Oversight",
        "content": (
            "Article 14 requires high-risk AI systems to be designed to allow effective "
            "human oversight during use. Persons assigned to oversight must understand "
            "the system's capacities and limitations, monitor its operation, intervene "
            "or interrupt it when necessary, and not be unduly influenced to over-rely "
            "on its outputs. Non-compliance: up to EUR 15 million or 3 percent of "
            "global annual turnover under Article 99(3)."
        )
    },
    "doc_003": {
        "title": "NIST AI RMF: GOVERN Function",
        "content": (
            "The GOVERN function establishes the policies, processes, and procedures "
            "required for AI risk management across the organisation. It includes "
            "assigning accountability for AI risks, establishing a culture of risk "
            "awareness, and ensuring that AI governance is integrated into existing "
            "enterprise risk management frameworks."
        )
    },
    "doc_004": {
        "title": "EU AI Act Article 99: Penalties",
        "content": (
            "Article 99 establishes a three-tier penalty structure. "
            "Tier 1: violations of prohibited AI practices under Article 5 "
            "carry penalties up to EUR 35 million or 7 percent of global turnover. "
            "Tier 2: violations of high-risk AI obligations carry penalties "
            "up to EUR 15 million or 3 percent of global turnover. "
            "Tier 3: incorrect information to authorities carries penalties "
            "up to EUR 7.5 million or 1 percent of global turnover."
        )
    },
    "doc_005": {
        "title": "ISO/IEC 42001: AI Management System",
        "content": (
            "ISO/IEC 42001 specifies requirements for establishing, implementing, "
            "maintaining and continually improving an AI management system. "
            "Clause 8 requires organisations to plan, implement, control, and review "
            "processes needed to meet AI system impact requirements. "
            "Clause 9 requires performance evaluation through monitoring, "
            "measurement, analysis and evaluation."
        )
    }
}

print(f"Knowledge base: {len(REGULATORY_DOCS)} regulatory documents loaded.")
for doc_id, doc in REGULATORY_DOCS.items():
    print(f"  {doc_id}: {doc['title']}")

Knowledge base: 5 regulatory documents loaded.
  doc_001: EU AI Act Article 10: Data Governance
  doc_002: EU AI Act Article 14: Human Oversight
  doc_003: NIST AI RMF: GOVERN Function
  doc_004: EU AI Act Article 99: Penalties
  doc_005: ISO/IEC 42001: AI Management System


In [14]:
import chromadb
import os

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/" # Added DRIVE_PATH definition
CHROMA_PATH = DRIVE_PATH + "chroma_p2/"
os.makedirs(CHROMA_PATH, exist_ok=True)

if not SIMULATED_OUTPUT:
    from chromadb.utils import embedding_functions
    embedding_fn = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
        api_key=userdata.get('GOOGLE_API_KEY'),
        model_name="models/embedding-001"
    )
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = chroma_client.get_or_create_collection(
        name="regulatory_docs_p2",
        embedding_function=embedding_fn
    )
    if collection.count() == 0:
        collection.add(
            documents=[d["content"] for d in REGULATORY_DOCS.values()],
            ids=list(REGULATORY_DOCS.keys()),
            metadatas=[{"title": d["title"]} for d in REGULATORY_DOCS.values()]
        )
    print(f"Chroma collection ready: {collection.count()} documents indexed.")
    print(f"Persisted to: {CHROMA_PATH}")
else:
    print("[SIMULATED] Chroma persistent client.")
    print(f"  Path: {CHROMA_PATH}")
    print(f"  Collection: regulatory_docs_p2")
    print(f"  Documents: {len(REGULATORY_DOCS)}")
    print(f"  Embedding model: models/embedding-001 (Google Generative AI)")
    print(f"  When live: semantic similarity replaces keyword matching.")

[SIMULATED] Chroma persistent client.
  Path: /content/drive/MyDrive/python-ai-governance-p2/data/chroma_p2/
  Collection: regulatory_docs_p2
  Documents: 5
  Embedding model: models/embedding-001 (Google Generative AI)
  When live: semantic similarity replaces keyword matching.


In [15]:
def retrieve_documents(query: str, n_results: int = 2) -> list:
    """Retrieve the most semantically relevant regulatory documents for a query.

    In simulated mode returns a representative result for a human oversight
    query. In live mode uses Chroma semantic similarity search.
    """
    if SIMULATED_OUTPUT:
        return [
            {
                "id": "doc_002",
                "title": REGULATORY_DOCS["doc_002"]["title"],
                "content": REGULATORY_DOCS["doc_002"]["content"],
                "distance": 0.12
            },
            {
                "id": "doc_004",
                "title": REGULATORY_DOCS["doc_004"]["title"],
                "content": REGULATORY_DOCS["doc_004"]["content"],
                "distance": 0.24
            }
        ]
    results = collection.query(query_texts=[query], n_results=n_results)
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "distance": results["distances"][0][i]
        }
        for i in range(len(results["ids"][0]))
    ]


def generate_response(query: str, retrieved_docs: list) -> dict:
    """Generate a grounded response using Gemini with retrieved regulatory context.

    Gemini is the system under test throughout this project. Claude is never
    used here. Claude's role is evaluator only, never generator.
    """
    context = "\n\n".join(
        f"[{d['title']}]\n{d['content']}" for d in retrieved_docs
    )
    prompt = (
        "You are a regulatory compliance assistant. "
        "Answer the following question using ONLY the information "
        "in the provided regulatory documents. "
        "If the answer is not in the documents, say so explicitly.\n\n"
        f"Documents:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )

    if SIMULATED_OUTPUT:
        return {
            "query": query,
            "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
            "response": (
                "Based on EU AI Act Article 14, high-risk AI systems must be "
                "designed to allow effective human oversight. Persons assigned "
                "to oversight must understand the system's capacities and "
                "limitations, monitor its operation, and be able to intervene "
                "or interrupt it when necessary. They must not be unduly "
                "influenced to over-rely on the system's outputs. "
                "Non-compliance with Article 14 carries penalties of up to "
                "EUR 15 million or 3 percent of global annual turnover "
                "under Article 99."
            ),
            "model": "gemini-flash-latest",
            "simulated": True
        }

    response = gemini_client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return {
        "query": query,
        "retrieved_doc_ids": [d["id"] for d in retrieved_docs],
        "response": response.text,
        "model": "gemini-flash-latest",
        "simulated": False
    }


print("retrieve_documents() defined.")
print("generate_response() defined.")
print("Gemini is the system under test. Claude is the evaluator. Never reversed.")

retrieve_documents() defined.
generate_response() defined.
Gemini is the system under test. Claude is the evaluator. Never reversed.


In [16]:
TEST_QUERY = (
    "What are the human oversight requirements for high-risk AI systems "
    "and what are the penalties for non-compliance?"
)

retrieved = retrieve_documents(TEST_QUERY)
result = generate_response(TEST_QUERY, retrieved)

print("=" * 60)
print(f"QUERY:\n{result['query']}")
print()
print("RETRIEVED DOCUMENTS:")
for doc_id in result['retrieved_doc_ids']:
    print(f"  {doc_id}: {REGULATORY_DOCS[doc_id]['title']}")
print()
print(f"RESPONSE:\n{result['response']}")
print()
print(f"MODEL: {result['model']}")
print(f"SIMULATED: {result['simulated']}")
print("=" * 60)

QUERY:
What are the human oversight requirements for high-risk AI systems and what are the penalties for non-compliance?

RETRIEVED DOCUMENTS:
  doc_002: EU AI Act Article 14: Human Oversight
  doc_004: EU AI Act Article 99: Penalties

RESPONSE:
Based on EU AI Act Article 14, high-risk AI systems must be designed to allow effective human oversight. Persons assigned to oversight must understand the system's capacities and limitations, monitor its operation, and be able to intervene or interrupt it when necessary. They must not be unduly influenced to over-rely on the system's outputs. Non-compliance with Article 14 carries penalties of up to EUR 15 million or 3 percent of global annual turnover under Article 99.

MODEL: gemini-flash-latest
SIMULATED: True


In [17]:
def create_trace(name: str, metadata: dict) -> dict:
    """Create a Langfuse trace for an evaluation run.

    Works in both simulated and live mode. In simulated mode the trace
    object is constructed locally but not transmitted to Langfuse.
    In live mode the trace is created in the Langfuse backend and the
    trace ID is returned for score attachment.

    All subsequent phases call this function before logging any scores.
    Langfuse tracing is established from Phase 1 so every evaluation
    run across all phases is observable from a single dashboard.
    """
    trace = {
        "name": name,
        "metadata": metadata,
        "scores": []
    }

    if not SIMULATED_OUTPUT:
        lf_trace = langfuse.trace(name=name, metadata=metadata)
        trace["langfuse_id"] = lf_trace.id
        print(f"Langfuse trace created: {lf_trace.id}")
    else:
        trace["langfuse_id"] = f"simulated-{name}"
        print(f"[SIMULATED] Trace: {trace['langfuse_id']}")

    return trace


def log_score(trace: dict, score_name: str,
              value: float, comment: str = "") -> None:
    """Log a named evaluation score to a Langfuse trace.

    Called after every metric computation in Phases 2 through 6.
    Score names follow a consistent convention across phases:
      phase_XX_metric_name
    This makes the Langfuse dashboard filterable by phase.
    """
    trace["scores"].append({
        "name": score_name,
        "value": round(value, 4),
        "comment": comment
    })

    if not SIMULATED_OUTPUT:
        langfuse.score(
            trace_id=trace["langfuse_id"],
            name=score_name,
            value=value,
            comment=comment
        )


print("create_trace() defined.")
print("log_score() defined.")
print("Both functions are reused in Phases 2 through 6.")
print("Score name convention: phase_XX_metric_name")

create_trace() defined.
log_score() defined.
Both functions are reused in Phases 2 through 6.
Score name convention: phase_XX_metric_name


In [18]:
baseline_trace = create_trace(
    name="phase01_baseline_rag",
    metadata={
        "phase": "01",
        "notebook": "01_environment_and_baseline",
        "query": TEST_QUERY,
        "retrieved_doc_ids": result["retrieved_doc_ids"],
        "model": result["model"],
        "simulated": SIMULATED_OUTPUT
    }
)

log_score(
    baseline_trace,
    "phase_01_retrieval_count",
    float(len(retrieved)),
    f"Retrieved {len(retrieved)} documents for baseline query"
)

log_score(
    baseline_trace,
    "phase_01_top_doc_distance",
    retrieved[0]["distance"],
    f"Cosine distance of top retrieved document: {retrieved[0]['id']}"
)

log_score(
    baseline_trace,
    "phase_01_response_length",
    float(len(result["response"])),
    "Character count of generated response"
)

print(f"Trace: {baseline_trace['langfuse_id']}")
print(f"Scores logged: {len(baseline_trace['scores'])}")
for s in baseline_trace["scores"]:
    print(f"  {s['name']}: {s['value']}  ({s['comment']})")

[SIMULATED] Trace: simulated-phase01_baseline_rag
Trace: simulated-phase01_baseline_rag
Scores logged: 3
  phase_01_retrieval_count: 2.0  (Retrieved 2 documents for baseline query)
  phase_01_top_doc_distance: 0.12  (Cosine distance of top retrieved document: doc_002)
  phase_01_response_length: 474.0  (Character count of generated response)


In [19]:
import json
from datetime import datetime

output = {
    "phase": "01_environment_and_baseline",
    "timestamp": datetime.now().isoformat(),
    "simulated": SIMULATED_OUTPUT,
    "tools": {
        "ragas": "0.2.0",
        "deepeval": "latest",
        "langfuse": "v4",
        "promptfoo": "0.121.19",
        "node": "v22.23.1",
        "chroma": "persistent client"
    },
    "knowledge_base": {
        "document_count": len(REGULATORY_DOCS),
        "doc_ids": list(REGULATORY_DOCS.keys()),
        "chroma_path": CHROMA_PATH
    },
    "baseline_test": {
        "query": result["query"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
        "top_doc_distance": retrieved[0]["distance"],
        "response_length_chars": len(result["response"]),
        "response_preview": result["response"][:200] + "..."
    },
    "langfuse_trace": {
        "trace_id": baseline_trace["langfuse_id"],
        "scores": baseline_trace["scores"]
    }
}

output_path = DRIVE_PATH + "phase01_baseline_results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved: {output_path}")
print()
print("Contents:")
print(f"  Tools verified: {len(output['tools'])}")
print(f"  Documents indexed: {output['knowledge_base']['document_count']}")
print(f"  Baseline query: {output['baseline_test']['query'][:60]}...")
print(f"  Scores logged: {len(output['langfuse_trace']['scores'])}")

Results saved: /content/drive/MyDrive/python-ai-governance-p2/data/phase01_baseline_results.json

Contents:
  Tools verified: 6
  Documents indexed: 5
  Baseline query: What are the human oversight requirements for high-risk AI s...
  Scores logged: 3


## Phase 1 Findings

**What was built:** A complete Project 2 evaluation environment with all four tools
installed and verified: RAGAS v0.2.0, DeepEval, Langfuse v4, and Promptfoo 0.121.19
running on Node v22.23.1. A baseline RAG pipeline using Gemini (gemini-flash-latest)
as the system under test, retrieving from a five-document regulatory corpus covering
EU AI Act Articles 10, 14, and 99, NIST AI RMF GOVERN function, and ISO/IEC 42001.
Langfuse tracing established from this first phase: create_trace() and log_score()
are defined here and reused in every subsequent phase so the full evaluation pipeline
is observable from a single dashboard.

**What was found:** The baseline pipeline retrieves correctly and generates grounded
responses. A test query on human oversight requirements retrieved doc_002
(Article 14) and doc_004 (Article 99), the two most relevant documents, and produced
a factually accurate, grounded response citing the correct penalty figures.
Three baseline scores were logged to the Langfuse trace: retrieval count (2),
top document distance (0.12), and response length.

**Simulated output note:** SIMULATED_OUTPUT = True throughout. All code is complete
and correctly structured. Simulated outputs match the exact schema of real API
responses. To run live: set SIMULATED_OUTPUT = False in Cell 5 and ensure all four
Colab secrets are set (GOOGLE_API_KEY, ANTHROPIC_API_KEY, LANGFUSE_PUBLIC_KEY,
LANGFUSE_SECRET_KEY). No other changes required in any cell.

**Known issue resolved:** Colab's default Node.js (v20.19.0) is below Promptfoo's
minimum requirement. Cell 4 upgrades Node to v22.23.1 via nvm before installing
Promptfoo. The run_promptfoo() helper function in Cell 4b ensures all Promptfoo
calls in Phase 5 use the correct Node version regardless of Colab's shell default.

**What this means:** Every subsequent phase evaluates this same baseline pipeline.
The architectural principle is in place before a single evaluation score is computed:
Gemini generates, Claude evaluates. Never reversed.

**Next step:** Phase 2a (02a_ragas_gemini_judge.ipynb): RAGAS evaluation of the
baseline pipeline using Gemini as the LLM judge to establish the same-family
baseline score. Phase 2b follows with Claude as judge to quantify the score
difference. The gap between those two scores is the same-family observer bias
finding, now measured rather than observed.